# System identification

**Your sim-tuned agent is slow, or crashes, on CARLA. Why?**

`agents/unit-07.py` was tuned on one track, `test-circuit`, with one car, racekit's carla-like car.
Every number in it is either a guess about the car (Unit 3's model: the throttle, the brake, the drag, the steering) or a knob that made it clean **there** (A_LAT = 4.6 m/s², A_DEC = 5.0 m/s²).
CARLA's Monza is a track it never saw, with a car whose numbers nobody gave you.

You do not need CARLA to see the problem.
`test-circuit-seed2026` is Checkpoint A's track: the same family as `test-circuit`, a different layout, and grip that differs section by section.
Race the Unit 7 agent at home and there (`carla_like(noise=True)`, seed 0):

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.runner import run_race
from racekit.sysid_data import load_log
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_07, UNIT_08 = AGENTS / "unit-07.py", AGENTS / "unit-08.py"
workdir = Path(mkdtemp(prefix="roar-unit-08-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
CLEAN_OFFSET = 2.0  # m: the clean rule's limit
DT = 0.05  # s: one control tick, one telemetry row


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # Unit 7's clean rule: finished, 0 collisions, and never more than 2 m off the line.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


def summary(result):
    if clean(result):
        return f"clean, {result.total_time:6.2f} s, max |offset| {max_offset(result):.2f} m"
    finished = "finished" if result.finished else "did not finish"
    return f"NOT clean: {result.collisions} collisions, {finished}, max |offset| {max_offset(result):.2f} m"


def race(source, track="test-circuit", seed=0, max_seconds=450.0, laps=3):
    # Every race of this lesson: the carla-like car with its noise, one seed.
    return run_race(source, track, car=carla_like(noise=True), seed=seed, max_seconds=max_seconds, laps=laps)


def row_sections(telemetry, track):
    # racekit's progress p is waypoint (p + 1) mod n (Unit 7): the grip section of every row.
    sections = np.array(grip_sections(track))
    return sections[(telemetry["progress"] + 1) % len(sections)]


home = race(UNIT_07)
away = race(UNIT_07, track="test-circuit-seed2026")
print("Unit 7 on test-circuit (its tuning track):", summary(home))
print("Unit 7 on test-circuit-seed2026          :", summary(away))
assert clean(home) and abs(home.total_time - 304.15) <= TICK
assert not clean(away)

checkpoint_track = load_track("test-circuit-seed2026")
held = away.telemetry["held"] == 1
crash_rows = [row for row in np.nonzero(held)[0] if row > 0 and not held[row - 1]]  # each hold's first row
section = row_sections(away.telemetry, checkpoint_track)
print("the section each crash happened in:", [int(section[row - 1]) for row in crash_rows])

fig, ax = plt.subplots(figsize=(6, 5))
points = ax.scatter(checkpoint_track.x, checkpoint_track.y, c=grip_sections(checkpoint_track), s=4, cmap="viridis")
fig.colorbar(points, ax=ax, label="grip section")
ax.scatter([away.telemetry["x"][row - 1] for row in crash_rows], [away.telemetry["y"][row - 1] for row in crash_rows],
           marker="x", color="red", label="a crash")
ax.set_aspect("equal")
ax.set_title("Unit 7 on test-circuit-seed2026: where it crashed")
ax.legend()
plt.show()

At home the agent is clean in **304.15 s**.
On `test-circuit-seed2026` it crashes over and over, and almost every crash is in **section 1**, the same corners every lap.
(Over seeds 0–4 it is not clean on any seed there; plan `roar-008`'s probe record, which a routed test re-races.)

It is not unlucky: it is **wrong about the car**.
Its planner asks for 4.6 m/s² of cornering everywhere, a number found just inside the clean edge on `test-circuit`, whose least grippy section measures 7.87 m/s² (§4).
Section 1 of this track measures 7.69 m/s², and a plan that sat just inside the edge at home is past it here.
On `monza-like` the same agent is clean but **slow**, 515.10 s (Unit 7), and this unit shows that a wrong steering number is part of why.

The cure is to stop guessing and **measure**.
This unit measures three things from logs of the car:

1. the **longitudinal map**: how hard the throttle pushes, the brake pulls and the drag holds back (Unit 3 guessed 6.0, 8.0 and 4.0e-4);
2. the **steering gain**: the wheel angle at full lock (Unit 3 guessed 70°);
3. the **grip** of each section, μ·g, which no log of a clean race can tell you, and which needs an experiment of its own.

The tool for the first two is **linear least squares**.

## 1. A log as a matrix

racekit keeps two **sensor logs** of the carla-like car, recorded the way CARLA records a car: what the driver **observed** each step (position, heading, the velocity vector in the world frame) and what it **commanded** (throttle, steer, brake), never the car's hidden state.
`racekit.sysid_data.load_log(name)` reads one: a table with the columns `t, x, y, yaw, vx, vy, throttle, steer, brake`, and a little meta.

Unit 3's longitudinal model says that the speed changes as

$$\frac{dv}{dt} = a_T\,\text{throttle} - a_B\,\text{brake} - c\,v|v|.$$

Each pair of consecutive log rows k → k + 1 gives one equation in the three unknowns (a_T, a_B, c):

$$\underbrace{\frac{v_{k+1} - v_k}{\Delta t}}_{b_k} \approx a_T \cdot \text{throttle}_k + a_B \cdot (-\text{brake}_k) + c \cdot (-\bar v_k |\bar v_k|),$$

with $\bar v_k$ the mean of the two speeds.
Stack the equations: row k of a matrix **A** holds the three numbers (throttle_k, −brake_k, −v̄_k|v̄_k|), and b holds the measured rates.
A race gives ten thousand equations for three unknowns.
No (a_T, a_B, c) satisfies them all (the log is noisy and the model is simple), so ask for the best one: the **least-squares** solution, the c that makes the sum of squared **residuals** ‖Ac − b‖² = Σ_k (A_k·c − b_k)² as small as it can be.
`np.linalg.lstsq(A, b, rcond=None)` returns it (its first output).

One more thing goes in the matrix.
A command does not act at once: Unit 5 met actuator delay.
If the car applies the command `lag` ticks late, then the speed change from k to k + 1 is driven by the command logged at k − lag.
The lag is unknown, so try several and let the residual choose.

In [ ]:
race_log, race_meta = load_log("monza-like-seed0.csv")
print(race_meta, f"{len(race_log)} rows, {race_log['t'][-1]:.2f} s")
race_speed = np.hypot(race_log["vx"], race_log["vy"])  # the speed is the length of the velocity vector


def longitudinal_rows(log, speed, lag):
    # One equation per pair of rows k -> k + 1 that are one tick apart (a respawn's hold is a clock gap),
    # driven by the command logged at k - lag. Returns (A, b, the rows k used).
    k = np.arange(lag, len(log) - 1)
    one_tick = np.abs(log["t"][k + 1] - log["t"][k] - DT) < 1e-6
    no_gap = np.abs(log["t"][k] - log["t"][k - lag] - lag * DT) < 1e-6
    k = k[one_tick & no_gap]
    mean = (speed[k] + speed[k + 1]) / 2.0
    a = np.column_stack([log["throttle"][k - lag], -log["brake"][k - lag], -mean * np.abs(mean)])
    b = (speed[k + 1] - speed[k]) / DT
    return a, b, k


def rms(a, b, c):
    # The root of the mean squared residual: the typical size of one equation's error (m/s^2).
    return float(np.sqrt(np.mean((a @ c - b) ** 2)))


fits = {}
for lag in range(4):
    a, b, _ = longitudinal_rows(race_log, race_speed, lag)
    c = np.linalg.lstsq(a, b, rcond=None)[0]
    fits[lag] = (c, rms(a, b, c))
    print(f"lag {lag}: a_T {c[0]:.3f}, a_B {c[1]:.3f}, c {c[2]:.3e}   residual RMS {fits[lag][1]:.3f} m/s^2")
best_lag = min(fits, key=lambda lag: fits[lag][1])
print("the lag with the smallest residual:", best_lag)
assert best_lag == 1
assert [round(fits[lag][1], 3) for lag in range(4)] == [0.570, 0.433, 0.535, 0.596]

The residual is smallest at **lag 1**: the speed change from k to k + 1 is best explained by the command logged one tick earlier.
Lag 0 and lag 2 both explain it worse, by a lot (0.570 and 0.535 against 0.433 m/s²): a residual is a measurement, and here it measures the car's delay.

**Is the fit real, or did it just fit the noise?**
Three numbers fitted to ten thousand equations can hardly overfit, but check it the honest way: fit on the first half of the race only, and measure the residual on the **second half**, rows the fit never saw.
Then put Unit 3's assumed map, (6.0, 8.0, 4.0e-4), on the same rows.

In [ ]:
a, b, k = longitudinal_rows(race_log, race_speed, 1)
first = race_log["t"][k] < race_log["t"][-1] / 2.0
c_half = np.linalg.lstsq(a[first], b[first], rcond=None)[0]
ASSUMED = np.array([6.0, 8.0, 4.0e-4])  # agents/unit-07.py: A_THROTTLE, A_BRAKE, C_DRAG
print(f"fitted on the first half : RMS {rms(a[first], b[first], c_half):.3f} there, "
      f"{rms(a[~first], b[~first], c_half):.3f} on the second half")
print(f"the assumed map          : RMS {rms(a[~first], b[~first], ASSUMED):.3f} on the second half")
assert rms(a[~first], b[~first], c_half) < rms(a[~first], b[~first], ASSUMED) - 0.1

c_fit = fits[1][0]
window = (race_log["t"][k] > 100.0) & (race_log["t"][k] < 115.0)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(race_log["t"][k][window], b[window], color="lightgrey", label="measured dv/dt")
ax.plot(race_log["t"][k][window], (a @ c_fit)[window], label="the fitted map")
ax.plot(race_log["t"][k][window], (a @ ASSUMED)[window], linestyle="--", label="the assumed map")
ax.set_xlabel("t (s)")
ax.set_ylabel("dv/dt (m/s^2)")
ax.set_title("monza-like-seed0.csv: 15 s of the race, and two models of it")
ax.legend()
plt.show()

The half-race fit is as good on the half it never saw (0.435 m/s²) as on its own (0.431): it learned the car, not the noise.
The assumed map is clearly worse on the same rows (0.556): in the plot it overshoots every full-throttle stretch.
That gap is the cost of guessing.

## 2. How least squares is solved: QR

`lstsq` is a black box so far.
Here is how a good solver finds the minimum, because Unit 9 opens the box to make a large problem small.

Every N × m matrix A with independent columns (here m = 3) can be written as **A = QR**:

- Q is N × m and its columns are **orthonormal**: each has length 1 and they are perpendicular, so QᵀQ = I (the m × m identity);
- R is m × m and **upper triangular**: zeros below the diagonal.

`np.linalg.qr(A)` returns the two.
Q's columns span the same directions as A's columns (A's **column space**).
Split b into its part inside that space, QQᵀb, and the part outside it, b − QQᵀb.
The two parts are perpendicular, and so, for **any** c,

$$Ac - b = Q(Rc - Q^{\mathsf T} b) - (b - QQ^{\mathsf T} b)$$

is a sum of two perpendicular vectors, and Pythagoras gives

$$\|Ac - b\|^2 = \|Rc - Q^{\mathsf T} b\|^2 + \text{const}, \qquad \text{const} = \|b - QQ^{\mathsf T} b\|^2 = \|b\|^2 - \|Q^{\mathsf T} b\|^2.$$

(‖Qy‖ = ‖y‖ because Q's columns are orthonormal.)
The constant is the part of b that **no** c can explain; c changes only the first term.
So the least-squares c makes the first term zero: it solves the small, triangular system **Rc = Qᵀb**.
A triangular system is solved from the bottom row up, one unknown at a time (**back substitution**): the last row has one unknown, the row above it has two, one of them already known, and so on.

Two things to take away: an N × m problem became an m × m one (ten thousand rows became three), and nothing was squared on the way, which keeps the arithmetic accurate (a solver that formed AᵀA would square the problem's sensitivity to rounding).

In [ ]:
q, r = np.linalg.qr(a)  # a and b: the lag-1 equations
print("Q:", q.shape, " R:", r.shape)
print("Q^T Q (the identity, to rounding):")
print(np.round(q.T @ q, 12))
print("R (upper triangular):")
print(np.round(r, 2))


def back_substitute(r, y):
    # Solve r c = y for an upper-triangular r, from the last row up.
    m = len(y)
    c = np.zeros(m)
    for i in range(m - 1, -1, -1):
        c[i] = (y[i] - np.sum(r[i, i + 1:] * c[i + 1:])) / r[i, i]
    return c


c_qr = back_substitute(r, q.T @ b)
print("QR + back substitution:", c_qr)
print("np.linalg.lstsq       :", c_fit)
assert np.max(np.abs(c_qr - c_fit) / np.abs(c_fit)) < 1e-9

constant = np.sum(b ** 2) - np.sum((q.T @ b) ** 2)
for name, c in (("the fit", c_fit), ("the assumed map", ASSUMED), ("all zeros", np.zeros(3))):
    left = np.sum((a @ c - b) ** 2)
    right = np.sum((r @ c - q.T @ b) ** 2) + constant
    print(f"{name:16s}: ||Ac - b||^2 = {left:10.2f}   ||Rc - Q^T b||^2 + const = {right:10.2f}")
    assert abs(left - right) < 1e-6 * left
print(f"const = {constant:.2f}: the fit's own sum of squared residuals, which no c can lower")

The identity holds at **every** c, not only at the answer, and at the answer the first term is zero, so the fit's sum of squares *is* the constant.
Unit 9's racing line is a least-squares problem with thousands of rows and over a hundred unknowns, and bounds on them; it uses exactly this step to hand a solver the small m × m problem instead of the tall one.

## 3. The car's numbers: the longitudinal map and the steering gain

**The longitudinal map** is the lag-1 fit (`monza-like-seed0.csv` is a full noisy race; plan `roar-008`'s routed test re-fits it):

| | measured (lag 1) | assumed by `agents/unit-07.py` |
|---|---|---|
| a_T, full throttle (m/s²) | 5.17 | 6.0 |
| a_B, full brake (m/s²) | 8.54 | 8.0 |
| c, drag (1/m) | 3.08e-4 | 4.0e-4 |
| delay | 1 tick | none in the model |

**The steering gain.**
Unit 3's kinematic bicycle turns a wheel angle δ into a yaw rate r = v·tan(δ) / L, with L = 2.875 m.
Turned round, the wheel angle the car **actually** had is δ = atan(r·L / v), and the submission commands δ = −steer · MAX_STEER.
So each row gives one equation, atan(r·L / v) ≈ MAX_STEER · (−steer): a one-column least-squares problem for MAX_STEER.

Two cautions decide which rows count.

- The yaw rate does not follow a **changing** steer at once: on the carla-like car the tyres need time to build up their force (Unit 6).
  The kinematic relation holds only in a **steady** turn, so a row counts only when the steer has been still: the steer logged over the last 11 ticks has a standard deviation below 0.002.
  The log for this is `oval-steady-seed0.csv`: one lap of the oval at a steady 15 m/s, long bends at an almost constant steer.
- The steer acts one tick late (§1's lag), so the yaw change from k to k + 1 is paired with the steer logged at k − 1.

The yaw rate is the heading change per tick, wrapped (Unit 2: a heading that crosses ±π must not jump by 2π).

In [ ]:
steady_log, steady_meta = load_log("oval-steady-seed0.csv")
print(steady_meta, f"{len(steady_log)} rows")
WHEELBASE = 2.875  # m, L (Unit 3)
yaw = steady_log["yaw"]
rate = ((yaw[1:] - yaw[:-1] + math.pi) % (2.0 * math.pi) - math.pi) / DT  # wrapped, over k -> k + 1
speed = np.hypot(steady_log["vx"], steady_log["vy"])
mean_speed = (speed[:-1] + speed[1:]) / 2.0
applied = np.concatenate([[0.0], steady_log["steer"][:-2]])  # for k -> k + 1, the steer logged at k - 1
stillness = np.array([np.std(applied[max(0, k - 10):k + 1]) for k in range(len(applied))])
steady = (mean_speed > 5.0) & (np.abs(applied) > 0.005) & (stillness < 0.002)
wheel_angle = np.array([math.atan2(r_k * WHEELBASE, v_k) for r_k, v_k in zip(rate[steady], mean_speed[steady])])
column = np.column_stack([-applied[steady]])  # one column: wheel_angle = MAX_STEER * (-steer)
gain = np.linalg.lstsq(column, wheel_angle, rcond=None)[0][0]
print(f"{int(np.sum(steady))} steady rows: measured MAX_STEER = {math.degrees(gain):.2f} deg (assumed: 70 deg)")
assert abs(math.degrees(gain) - 65.71) < 0.005

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(-applied[steady], np.degrees(wheel_angle), s=4, color="grey", label="steady rows")
line = np.linspace(0.0, np.max(-applied[steady]), 2)
ax.plot(line, math.degrees(gain) * line, label=f"measured: {math.degrees(gain):.1f} deg x (-steer)")
ax.plot(line, 70.0 * line, linestyle="--", label="assumed: 70 deg x (-steer)")
ax.set_xlabel("-steer (the command, one tick earlier)")
ax.set_ylabel("wheel angle atan(r L / v) (deg)")
ax.set_title("oval-steady-seed0.csv: the steering gain")
ax.legend()
plt.show()

**Measured 65.7°, against the 70° the agent assumed.**
Every steering command the agent sends turns the wheels about 6% less than its controller intends.
On a corner pure pursuit corrects for it soon enough; on a long, fast straight the late, weak corrections can grow into a weave, and §5 shows that this is exactly what limits the speed on `monza-like`.
`agents/unit-08.py` sets `MAX_STEER = math.radians(66.0)`: the measurement rounded to a whole degree, with a comment that says so.

The longitudinal numbers stay as they are in the submission: they feed only Unit 3's 1-second prediction, which the agent measures but does not use to drive.
(Unit 10's model predictive control will drive with a model, and then they matter.)

## 4. Grip: the limit probe

**Why a race log cannot tell you the grip.**
A clean race never asks the tyres for everything they have: the planner asks for A_LAT, and A_LAT is safe *because* it is well inside the limit.
Look at how hard the car actually accelerated in the home race at the top of this lesson.
The telemetry has the car's position every 0.05 s, so its acceleration vector is the **second difference** of the positions,

$$\vec a_k \approx \frac{\vec p_{k+1} - 2\vec p_k + \vec p_{k-1}}{\Delta t^2},$$

braking, accelerating and cornering all at once, and |a_k| is its length.
A row counts when it and its two neighbours are driving rows (`held == 0`), neither step jumps more than 5 m (a respawn teleports the car), and the car is above 5 m/s.
Per grip section, take a high percentile of |a|: the **99th** (`np.percentile(values, 99)`), a "nearly the largest" that one wild row cannot move.

In [ ]:
MIN_ROWS = 20  # a section with this many qualifying rows or fewer gives no estimate
PERCENTILE = 99.0


def accel_rows(telemetry):
    # |a| of the middle row of every three consecutive rows (rows 1 .. N - 2), and which of them qualify.
    x, y, held = telemetry["x"], telemetry["y"], telemetry["held"]
    ax_rows = (x[2:] - 2.0 * x[1:-1] + x[:-2]) / DT ** 2
    ay_rows = (y[2:] - 2.0 * y[1:-1] + y[:-2]) / DT ** 2
    step = np.hypot(np.diff(x), np.diff(y))
    ok = (held[:-2] == 0) & (held[1:-1] == 0) & (held[2:] == 0)
    ok = ok & (step[:-1] < 5.0) & (step[1:] < 5.0) & (telemetry["v"][1:-1] > 5.0)
    return np.hypot(ax_rows, ay_rows), ok


def section_accel(telemetry, track):
    # Per grip section: (qualifying rows, the 99th percentile of |a|, or None with MIN_ROWS rows or fewer).
    accel, ok = accel_rows(telemetry)
    section = row_sections(telemetry, track)[1:-1]
    table = []
    for s in range(6):
        values = accel[ok & (section == s)]
        table.append((len(values), float(np.percentile(values, PERCENTILE)) if len(values) > MIN_ROWS else None))
    return table


test_circuit = load_track("test-circuit")
home_accel = section_accel(home.telemetry, test_circuit)
print("Unit 7's clean home race, 99th percentile of |a| per section (m/s^2):",
      [round(value, 2) for _, value in home_accel])

Between 6.9 and 7.7 m/s² in every section.
Is that the grip?
It cannot be told from this log.
The car never slid, so all these numbers say is that the tyres **gave at least this much**: μ·g ≥ 7.73 m/s² in section 1, and so on.
A clean race measures the driver's plan, not the tyres' limit; it gives a **lower bound** and nothing more.

**The limit probe.**
To measure a limit you must reach it, and you can choose where.
Take your own Unit 7 agent and make it gentle everywhere, `A_LAT = 3.5`, except in **one** section s, whose knob `GRIP[s] = 3.5` lets the planner ask for 3.5 × 3.5 = 12.25 m/s² of cornering there, far more than any section of these tracks turned out to give (the table below).
In section s the car arrives at the corners too fast, slides, and usually crashes; the slide happens **at the limit**, and the positions record it.
Elsewhere it drives gently, so the lap goes on and the next corner of section s is probed too.
One lap (`laps=1`), `max_seconds=300`, seed 0; six races per track, one per section.
The estimate of μ·g in section s is the 99th percentile of |a| over section s's qualifying rows **of the race that pushed s**.

Why one section at a time?
The probe record (plan `roar-008`) tried the two obvious shortcuts, and both fail: pushing every section at once crashes in the first corner, again and again, and pushing every section from s onward crashes in an earlier low-grip section and never reaches the later ones at the limit.

In [ ]:
PROBE_A_LAT, PROBE_KNOB = 3.5, 3.5


def probe(track, section):
    # The student's own Unit 7 agent, gentle everywhere except one pushed section; one lap.
    grip = [1.0] * 6
    grip[section] = PROBE_KNOB
    source = variant(UNIT_07, f"probe-{track}-{section}", A_LAT=PROBE_A_LAT, GRIP=tuple(grip))
    return race(source, track=track, max_seconds=300.0, laps=1)


test_circuit_probes = [probe("test-circuit", s) for s in range(6)]
mu_g_hat = []
print("test-circuit, one probe per section:")
for s, result in enumerate(test_circuit_probes):
    rows, value = section_accel(result.telemetry, test_circuit)[s]
    mu_g_hat.append(round(value, 2))
    print(f"  section {s}: {result.collisions:2d} collisions, {rows:4d} qualifying rows, mu * g = {value:.2f} m/s^2")
print("measured mu * g (m/s^2), rounded to 0.01:", tuple(mu_g_hat))
assert tuple(mu_g_hat) == (7.89, 9.72, 8.65, 7.87, 8.30, 8.14)  # agents/unit-08.py's MU_G
assert all(result.collisions >= 1 for result in test_circuit_probes)  # every probe slid: real estimates
assert all(home_value < probe_value for (_, home_value), probe_value in zip(home_accel, mu_g_hat))

pushed = test_circuit_probes[1]
accel, ok = accel_rows(pushed.telemetry)
section = row_sections(pushed.telemetry, test_circuit)[1:-1]
t = pushed.telemetry["t"][1:-1]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(t[ok & (section != 1)], accel[ok & (section != 1)], ".", markersize=2, color="lightgrey",
        label="other sections (gentle)")
ax.plot(t[ok & (section == 1)], accel[ok & (section == 1)], ".", markersize=2, label="section 1 (pushed)")
ax.axhline(mu_g_hat[1], color="black", linewidth=1.0, label=f"the 99th percentile in section 1: {mu_g_hat[1]:.2f}")
ax.set_xlabel("t (s)")
ax.set_ylabel("|a| (m/s^2)")
ax.set_title("the probe of section 1 on test-circuit")
ax.legend()
plt.show()

Every probe crashed (3 to 17 times), so in every section the car **did** reach the limit, and each number is an **estimate**, not a bound.
They range from 7.87 to 9.72 m/s²: the grip really does differ by section, by about 25%, and each probe's number lies above the clean race's in its section.
These six numbers are `MU_G` in `agents/unit-08.py`.

**When a probe proves nothing.**
A probe is only an estimate where the car actually lost grip.
On `monza-like`, section 0 is the start straight: there is no corner in it fast enough to slide, whatever the planner asks.

In [ ]:
monza = load_track("monza-like")
start_probe = probe("monza-like", 0)
rows, value = section_accel(start_probe.telemetry, monza)[0]
print(f"monza-like, section 0 pushed: {start_probe.collisions} collisions, {rows} qualifying rows, "
      f"99th percentile {value:.2f} m/s^2")
assert start_probe.collisions == 0 and round(value, 2) == 5.38

No collision: the car never slid in section 0, so **5.38 m/s² is a lower bound**, not an estimate.
The rules of the probe, in full:

- a section whose probe **crashed** gives an **estimate**, the 99th percentile;
- a section whose probe **never crashed** gives a **lower bound** (the grip is at least this); the planner may use it as measured, because it can only be too cautious;
- a section with **20 qualifying rows or fewer** gives **no estimate** at all: too few rows for a 99th percentile to mean anything.

The probe record has all seven tracks (seed 0, rounded to 0.01 m/s²; a routed test re-races `test-circuit`'s and `monza-like`'s probes, and the milestone `test-circuit`'s and `monza-like`'s section 0):

| track | s0 | s1 | s2 | s3 | s4 | s5 |
|---|---|---|---|---|---|---|
| `test-circuit` | 7.89 | 9.72 | 8.65 | 7.87 | 8.30 | 8.14 |
| `test-circuit-seed1` | 7.72 | 9.80 | 9.36 | 8.38 | 9.53 | 7.56 |
| `test-circuit-seed2` | 9.06 | 8.37 | 8.49 | 8.86 | 9.78 | 9.92 |
| `test-circuit-seed3` | 9.55 | 8.89 | 8.15 | 9.94 | 8.50 | 9.09 |
| `test-circuit-seed4` | 8.79 | 8.53 | 9.10 | 8.54 | 8.41 | 9.97 |
| `test-circuit-seed2026` | 7.70 | **7.69** | 7.70 | 9.81 | 8.16 | 8.04 |
| `monza-like` | 5.38 (a lower bound) | 8.96 | 9.12 | 9.14 | 9.70 | 7.97 |

Every probe of the six `test-circuit` tracks crashed (3 to 18 times); on `monza-like` sections 1–5 crashed and section 0 did not.
Look at section 1 of `test-circuit-seed2026`, where the hook's crashes were: 7.69 m/s², against 9.72 in section 1 of `test-circuit`.
All 42 probe races take about 20 s on eight cores; one lap each is cheap, which is why the probe is a practical tool and not just an idea.

## 5. Using what was measured

The submission now knows two things it guessed before.
`agents/unit-08.py` changes Unit 7's corner cap to

$$\bar v_i = \min\left(v_{\text{top}},\ \sqrt{\frac{f \cdot \hat\mu_s g \cdot \text{GRIP}[s]}{|\kappa_i|}}\right),$$

with $\hat\mu_s g$ = `MU_G[s]`, the measured grip of waypoint i's section, and f = `MU_FRACTION`, **one** number: the share of the measured grip the planner uses.
Everything else is Unit 7's (the forward–backward profile, the lead, pure pursuit), and `GRIP` is still a speed knob on top, shipped at 1.
With `MU_G = None` the cap is Unit 7's √(A_LAT·GRIP[s] / |κ_i|) again.
(Unit 7's A_LAT = 4.6 on `test-circuit` was f·μ̂g in disguise: 4.6 / 7.87 = 0.58 of its lowest section.)

`MU_G` is per track, like Unit 7's knobs; for another track, the probe gives its six numbers and `write_variant` writes them in, exactly as Checkpoint A's settings were written in.

**The steering gain matters (U8.3).**
On `monza-like`, give the measured-grip planner the track's own `MU_G` and f = 0.54, and race it with the assumed 70° and with the measured 66°.
The probe record says that at 70° it is not clean on 4 of seeds 0–4; seed 1 is one of them:

In [ ]:
MONZA_MU_G = (5.38, 8.96, 9.12, 9.14, 9.70, 7.97)
at_70 = race(variant(UNIT_08, "monza-70deg", MU_G=MONZA_MU_G, MU_FRACTION=0.54, MAX_STEER=math.radians(70.0)),
             track="monza-like", seed=1, max_seconds=900.0)
at_66 = race(variant(UNIT_08, "monza-66deg", MU_G=MONZA_MU_G, MU_FRACTION=0.54), track="monza-like", seed=1,
             max_seconds=900.0)
print("monza-like, f = 0.54, seed 1, MAX_STEER 70 deg:", summary(at_70))
print("monza-like, f = 0.54, seed 1, MAX_STEER 66 deg:", summary(at_66))
assert not clean(at_70)
assert clean(at_66) and abs(at_66.total_time - 510.15) <= TICK

unit_7_at_66 = race(variant(UNIT_07, "unit-07-66deg", MAX_STEER=math.radians(66.0)), track="test-circuit-seed2026")
print("Unit 7 with only the steering gain fixed, test-circuit-seed2026:", summary(unit_7_at_66))
assert not clean(unit_7_at_66)

With the assumed gain the car weaves on the long straights, more than 2 m off the line, and hits the wall; with the measured gain the same planner is clean, in 510.15 s, already faster than Unit 7's 515.10 s.
Over seeds 0–4 (probe record, plan `roar-008` U8.3, asserted by a routed test; the milestone races its edge):

| `monza-like`, measured grip | f = 0.54 | 0.56 | 0.58 | 0.60 | 0.62 | 0.64 |
|---|---|---|---|---|---|---|
| MAX_STEER 70° (assumed) | not clean | not clean | not clean | — | — | — |
| MAX_STEER 66° (measured) | 510.20 s | 506.50 s | 503.45 s | 500.70 s | **497.90 s** | not clean |

At 70° the steering, not the grip, is the limit; at 66° the planner is clean up to 0.62, 17 s faster than Unit 7.
The last race above is the other half of the story: on `test-circuit-seed2026` fixing the steering alone does **not** save Unit 7 (not clean on any of seeds 0–4).
There the problem is the grip, and only the measured grip fixes it.

**One fraction for the family (U8.4).**
Tune f the way Unit 7 tuned A_LAT, on the **training family**, the five `test-circuit` tracks, each with its own probed `MU_G`, at 66°, seeds 0–4 (probe record, plan `roar-008` U8.4; medians in s, spreads ≤ 0.25 s):

| f | `test-circuit` | seed1 | seed2 | seed3 | seed4 |
|---|---|---|---|---|---|
| 0.56 | 301.95 | 297.65 | 297.00 | 297.45 | 298.40 |
| 0.58 | 299.30 | 295.15 | 294.35 | 294.75 | 295.80 |
| 0.60 | 296.75 | 292.65 | 291.85 | 292.20 | not clean |

The family's edge is **0.58**: 0.60 is not clean on `test-circuit-seed4`.
Unit 7's **margin rule** ships one grid step inside the edge: **f = 0.56**.
Here is the edge on `test-circuit-seed4`, seed 0:

In [ ]:
SEED4_MU_G = (8.79, 8.53, 9.10, 8.54, 8.41, 9.97)
edge = race(variant(UNIT_08, "seed4-0.58", MU_G=SEED4_MU_G, MU_FRACTION=0.58), track="test-circuit-seed4")
past_edge = race(variant(UNIT_08, "seed4-0.60", MU_G=SEED4_MU_G, MU_FRACTION=0.60), track="test-circuit-seed4")
print("test-circuit-seed4, f = 0.58:", summary(edge))
print("test-circuit-seed4, f = 0.60:", summary(past_edge))
assert clean(edge) and abs(edge.total_time - 295.75) <= TICK
assert not clean(past_edge)

**The transfer.**
Now the question Checkpoint A asked: does a setting tuned on the family work on a track it never saw?
`test-circuit-seed2026` played no part in choosing f.
On it, the procedure is: probe its six sections (the table in §4), write its `MU_G` in, and race with the frozen f = 0.56.

In [ ]:
CHECKPOINT_MU_G = (7.70, 7.69, 7.70, 9.81, 8.16, 8.04)
transfer = race(variant(UNIT_08, "seed2026-0.56", MU_G=CHECKPOINT_MU_G), track="test-circuit-seed2026")
print("test-circuit-seed2026, f = 0.56 (frozen on the family), seed 0:", summary(transfer))
assert clean(transfer) and abs(transfer.total_time - 303.25) <= TICK

Clean, on the seed where Unit 7 crashed again and again.
Over seeds 0–4 it takes **303.30 s** (spread 0.10 s; probe record, plan `roar-008` U8.4, asserted by the milestone), against **310.80 s** for Checkpoint A's tuned global setting on the same track (spread 0.20 s): 7.5 s faster, far beyond the noise, since even 303.30 + 3 × 0.20 s (three times the larger spread) is below 310.80.
The probe record also has 0.58 clean on `test-circuit-seed2026` and 0.60 not clean there, the same edge as on the family: the measured grip carried the track's difference, so one fraction could carry the rest.

That is the unit's gate: **one fraction of the measured grip, tuned on the training family, transfers to the unseen track, faster than Checkpoint A.**

**The reference (U8.5).**
`agents/unit-08.py` ships `test-circuit`'s `MU_G`, f = 0.56 and 66°:

In [ ]:
reference = race(UNIT_08)
print("agents/unit-08.py on test-circuit, seed 0:", summary(reference))
assert clean(reference) and abs(reference.total_time - 301.95) <= TICK
print(f"against Unit 7 at home: {home.total_time - reference.total_time:.2f} s faster")

301.95 s at home (median 301.95 s over seeds 0–4), a little faster than Unit 7's 304.15 s, but that was never the point: Unit 7 was tuned **on** this track.
The point is that the same procedure, probe and then plan with one frozen fraction, also works where Unit 7's tuning does not.
With `MU_G = None`, the file races exactly like `agents/unit-07.py` with a 66° gain (the milestone checks it, to the tick).

| on the carla-like car | `test-circuit` | `test-circuit-seed2026` (unseen) | `monza-like` |
|---|---|---|---|
| Unit 7 (70°, A_LAT 4.6) | 304.15 s | not clean | 515.10 s |
| Checkpoint A (tuned global setting) | | 310.80 s | |
| Unit 8 (66°, f = 0.56 of the measured grip) | 301.95 s | **303.30 s** | 506.50 s |

## 6. To the real car

Everything above is racekit's carla-like car, whose logs come from racekit.
The lab of this unit (`labs/real-sim.ipynb`) does the same identification on **CARLA**: it records your Unit 7 agent on Monza and one steady lap, fits the longitudinal map and the steering gain on **your** logs, runs the six grip probes there with the same rules (estimate, lower bound, no estimate), predicts with racekit first, and then races a short, fixed ladder of fractions, accepting only a value it raced clean twice.

What carries over to every later unit:

- a model's numbers are **measured**, from logs, with least squares, and the residual chooses what the model leaves open (the lag);
- a fit is checked on **rows it did not use**;
- a limit is measured by **reaching it**, one place at a time; where it was not reached, the number is a lower bound and is called one;
- one global knob (f) on top of measured per-place numbers **transfers** where per-track tuning does not.

Next unit: the racing line, where the centreline stops being the path, and the speed tuning has to be checked again.